# String Operations

**Learning Objectives:**
- Use the `.str` accessor for vectorized string methods
- Apply string methods: `lower`, `upper`, `strip`, `split`, `replace`, `contains`, `extract`
- Use regex patterns in `extract()` and `extractall()`
- Concatenate strings with `cat()`
- One-hot encode with `get_dummies()`
- Clean messy text data

## 1. Setup – Messy Text Data

Let's create a realistic messy dataset.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["  Alice Smith  ", "BOB JONES", "charlie brown", "DIANA  ", "  eva"],
    "Email": ["alice@example.com", "bob@EXAMPLE.ORG", "charlie@gmail", "diana@", "eva@company.org"],
    "Phone": ["212-555-0100", "(312) 555-0201", "555.0302", "4155550403", "NULL"],
    "Category": ["A", "B", "A,B", "A|B|C", "a "]
})
print("Raw data:")
print(df)

## 2. The `.str` Accessor

The `.str` accessor gives every string method without writing a loop.

In [ ]:
# Clean names: strip, title case
df["Name_Clean"] = df["Name"].str.strip().str.title()
print("Cleaned names:")
print(df[["Name", "Name_Clean"]])

In [ ]:
# Case conversion
print("Lower:", df["Email"].str.lower().values)
print("Upper:", df["Name"].str.strip().str.upper().values)

## 3. `contains()` – Pattern Detection

Returns a boolean Series indicating if the pattern is found.

In [ ]:
# Which emails contain 'example'?
print("Contains 'example':")
print(df["Email"].str.contains("example", case=False))

# Valid email check (crude regex)
has_at = df["Email"].str.contains("@")
has_dot = df["Email"].str.contains(r"\.\w+", na=False)
print("\nHas @ and domain:", has_at & has_dot)

## 4. `extract()` and `extractall()` – Regex Capture Groups

Pull out specific parts of a string using regex groups `()`.

In [ ]:
# Extract username and domain from emails
extracted = df["Email"].str.extract(r"(\w+)@(.+)", expand=True)
extracted.columns = ["Username", "Domain"]
print("Email parts:")
print(extracted)

# Extract phone area codes
df["AreaCode"] = df["Phone"].str.extract(r"(\d{3})")
print("\nArea codes:")
print(df[["Phone", "AreaCode"]])

In [ ]:
# extractall() – multiple matches per row
df_multi = pd.DataFrame({"tags": ["a,b,c", "d,e", "f"]})
print("Multiple matches:")
print(df_multi["tags"].str.extractall(r"(\w)"))

## 5. `split()` and `cat()`

Split strings into multiple columns or concatenate them.

In [ ]:
# Split category (handling different delimiters)
categories = df["Category"].str.split(r"[,|]")
print("Split categories:")
print(categories)

# Concatenate strings
df["FullInfo"] = df["Name_Clean"].str.cat(df["Email"], sep=" <", na_rep="N/A") + ">"
print("\nConcatenated info:")
print(df["FullInfo"])

## 6. `get_dummies()` – One-Hot Encoding

Convert categorical strings into indicator columns.

In [ ]:
# One-hot encode a column
encoded = pd.get_dummies(df["Category"].str.strip().str.upper())
print("One-hot encoded categories:")
print(encoded)

# For multi-label categories, split first then explode + encode
cats = df["Category"].str.replace(",", "|").str.split("|")
exploded = df[["Name_Clean"]].copy()
exploded["Cat"] = cats
exploded = exploded.explode("Cat")
exploded["Cat"] = exploded["Cat"].str.strip().str.upper()
dummies = pd.get_dummies(exploded, columns=["Cat"]).groupby("Name_Clean").sum()
print("\nMulti-label one-hot:")
print(dummies)

## 7. Other Useful String Methods

In [ ]:
# replace
phone_clean = df["Phone"].str.replace(r"[^\d]", "", regex=True)
print("Cleaned phones:", phone_clean.values)

# len
print("\nName lengths:", df["Name"].str.strip().str.len().values)

# startswith / endswith
print("Starts with 'a':", df["Email"].str.startswith("a").values)

# count
print("Count of 'a':", df["Email"].str.count("a").values)

## 8. Practice Problems

In [ ]:
# Problem 1: From the Email column, extract all valid-looking email domains.
#            A valid domain should have a dot (e.g., example.com, company.org).
df["Valid_Domain"] = df["Email"].str.extract(r"@([\w.-]+\.[a-zA-Z]{2,})")
print(df[["Email", "Valid_Domain"]])

In [ ]:
# Problem 2: Clean the Phone column: strip non-digit characters, output as
#            a single string of digits. Also flag rows where phone was 'NULL'.
df["Phone_Clean"] = df["Phone"].str.replace(r"[^\d]", "", regex=True)
df["Phone_Missing"] = df["Phone"].str.upper().str.contains("NULL")
print(df[["Phone", "Phone_Clean", "Phone_Missing"]])

In [ ]:
# Problem 3: One-hot encode the cleaned Name_Clean column into a
#            multi-label format where each word is a category.
#            (Alice Smith -> Alice, Smith)
name_parts = df["Name_Clean"].str.split()
names_exploded = df["Name_Clean"].to_frame()
names_exploded["Part"] = name_parts
names_exploded = names_exploded.explode("Part")
name_dummies = pd.get_dummies(names_exploded.set_index("Name_Clean")["Part"]).groupby(level=0).sum()
print(name_dummies)

## 9. Summary

| Method | Purpose |
|--------|---------|
| `.str.lower/upper/title()` | Case conversion |
| `.str.strip()` | Remove whitespace |
| `.str.contains()` | Check pattern presence |
| `.str.extract()` | Regex capture groups |
| `.str.split()` | Split into multiple columns |
| `.str.cat()` | Concatenate strings |
| `.str.replace()` | Regex-based replacement |
| `get_dummies()` | One-hot encode categories |